In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, classification_report, 
                              confusion_matrix, roc_auc_score)
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

In [ ]:
# Load clean data and World Cups data
df = pd.read_csv("../data/processed/clean_data.csv")
cups = pd.read_csv("../data/raw/WorldCups.csv")

print(f"Clean data: {df.shape}")
print(f"World Cups: {cups.shape}")

In [ ]:
# Standardize team names
team_name_mapping = {
    "Germany FR": "Germany",
    "USA": "United States",
    "Soviet Union": "Russia",
    'rn">Germany': "Germany",
    'rn">Republic of Ireland': "Republic of Ireland",
    'rn">Serbia and Montenegro': "Serbia and Montenegro",
    'rn">Trinidad and Tobago': "Trinidad and Tobago"
}

df["team"] = df["team"].replace(team_name_mapping)

for column in ["Winner", "Runners-Up", "Third", "Fourth"]:
    cups[column] = cups[column].replace(team_name_mapping)

In [ ]:
# Create target: reached_final (Winner or Runner-Up = 1, else = 0)
final_columns = ["Winner", "Runners-Up"]

finalists = cups.melt(
    id_vars="Year",
    value_vars=final_columns,
    value_name="team"
)[["Year", "team"]].drop_duplicates()

finalists["reached_final"] = 1

# Merge with main data
df = df.merge(
    finalists,
    on=["Year", "team"],
    how="left"
)

df["reached_final"] = df["reached_final"].fillna(0).astype(int)

print("Target distribution:")
print(df["reached_final"].value_counts())
print(df["reached_final"].value_counts(normalize=True).round(3))

In [ ]:
# Add confederation
confederation_mapping = {
    "Argentina": "CONMEBOL", "Brazil": "CONMEBOL", "Uruguay": "CONMEBOL",
    "Chile": "CONMEBOL", "Colombia": "CONMEBOL", "Paraguay": "CONMEBOL",
    "Peru": "CONMEBOL", "Ecuador": "CONMEBOL", "Bolivia": "CONMEBOL",
    "Germany": "UEFA", "France": "UEFA", "Spain": "UEFA", "Italy": "UEFA",
    "England": "UEFA", "Netherlands": "UEFA", "Portugal": "UEFA",
    "Belgium": "UEFA", "Croatia": "UEFA", "Switzerland": "UEFA",
    "Sweden": "UEFA", "Denmark": "UEFA", "Poland": "UEFA", "Russia": "UEFA",
    "Mexico": "CONCACAF", "United States": "CONCACAF", "Costa Rica": "CONCACAF",
    "Canada": "CONCACAF", "Japan": "AFC", "Korea Republic": "AFC",
    "Iran": "AFC", "Saudi Arabia": "AFC", "Australia": "AFC",
    "Cameroon": "CAF", "Nigeria": "CAF", "Ghana": "CAF",
    "Morocco": "CAF", "Senegal": "CAF", "South Africa": "CAF",
    "New Zealand": "OFC"
}

df["confederation"] = df["team"].map(confederation_mapping).fillna("Other")

In [ ]:
# Prepare features
numeric_features = [
    "matches_played", "goals_for", "goals_against", "wins", "draws", "losses",
    "average_attendance", "goals_per_match", "win_percentage", "goal_difference"
]

categorical_features = ["confederation"]
target = "reached_final"

model_data = df[["Year", "team"] + numeric_features + categorical_features + [target]].dropna().copy()

print(f"Model data: {model_data.shape}")
print(f"\nTarget distribution:")
print(model_data[target].value_counts())
print(model_data[target].value_counts(normalize=True).round(3))

In [ ]:
# One-Hot Encoding
features = pd.get_dummies(
    model_data[numeric_features + categorical_features],
    columns=categorical_features,
    drop_first=False
)

features[target] = model_data[target].values

FEATURES = [col for col in features.columns if col != target]

print(f"Features: {len(FEATURES)}")
print(FEATURES)

In [ ]:
# Split and scale
X = features[FEATURES]
y = features[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled = pd.DataFrame(X_train_scaled, columns=FEATURES)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=FEATURES)

print(f"Train: {X_train_scaled.shape}")
print(f"Test: {X_test_scaled.shape}")
print(f"\nTarget (train): {y_train.value_counts().to_dict()}")
print(f"Target (test): {y_test.value_counts().to_dict()}")

In [ ]:
# Model 1: Logistic Regression
model_lr = LogisticRegression(random_state=42, max_iter=1000)
model_lr.fit(X_train_scaled, y_train)
pred_lr = model_lr.predict(X_test_scaled)
prob_lr = model_lr.predict_proba(X_test_scaled)[:, 1]

print("=" * 50)
print("LOGISTIC REGRESSION")
print("=" * 50)
print(f"Accuracy: {accuracy_score(y_test, pred_lr):.3f}")
print(f"ROC-AUC:  {roc_auc_score(y_test, prob_lr):.3f}")
print("\nReport:")
print(classification_report(y_test, pred_lr, target_names=["No final", "Final"]))

In [ ]:
# Model 2: Random Forest
model_rf = RandomForestClassifier(
    n_estimators=100, 
    max_depth=5, 
    random_state=42
)
model_rf.fit(X_train_scaled, y_train)
pred_rf = model_rf.predict(X_test_scaled)
prob_rf = model_rf.predict_proba(X_test_scaled)[:, 1]

print("=" * 50)
print("RANDOM FOREST")
print("=" * 50)
print(f"Accuracy: {accuracy_score(y_test, pred_rf):.3f}")
print(f"ROC-AUC:  {roc_auc_score(y_test, prob_rf):.3f}")
print("\nReport:")
print(classification_report(y_test, pred_rf, target_names=["No final", "Final"]))

In [ ]:
# Feature importance (Random Forest)
importances = pd.DataFrame({
    "feature": FEATURES,
    "importance": model_rf.feature_importances_
}).sort_values("importance", ascending=False)

print("Feature Importance:")
print(importances)

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=importances, x="importance", y="feature", color="#2563eb", ax=ax)
ax.set_title("Feature Importance - Random Forest", fontsize=14)
plt.tight_layout()
plt.savefig("../outputs/figures/08_feature_importance.png", dpi=150)

In [ ]:
# Compare and select best model
acc_lr = accuracy_score(y_test, pred_lr)
acc_rf = accuracy_score(y_test, pred_rf)

if acc_rf >= acc_lr:
    best_model = model_rf
    model_name = "Random Forest"
else:
    best_model = model_lr
    model_name = "Logistic Regression"

# Save model and scaler
joblib.dump(best_model, "../outputs/final_model.pkl")
joblib.dump(scaler, "../outputs/scaler.pkl")

print(f"Best model: {model_name}")
print(f"Accuracy: {max(acc_lr, acc_rf):.3f}")
print(f"Saved to: outputs/final_model.pkl")

In [ ]:
# Save processed data
X_train_scaled.to_csv("../data/processed/X_train.csv", index=False)
X_test_scaled.to_csv("../data/processed/X_test.csv", index=False)
y_train.to_csv("../data/processed/y_train.csv", index=False)
y_test.to_csv("../data/processed/y_test.csv", index=False)

print(f"Data saved to data/processed/")
print(f"  X_train.csv: {X_train_scaled.shape}")
print(f"  X_test.csv: {X_test_scaled.shape}")
print(f"  y_train.csv: {y_train.shape}")
print(f"  y_test.csv: {y_test.shape}")